INITIAL SETUP

In [ ]:
!pip install -q transformers accelerate gradio

In [ ]:
import time
import torch
import gradio as gr

from transformers import AutoTokenizer, AutoModelForCausalLM

print("LLM CHATBOT USING PRE-TRAINED TRANSFORMER")
print("PyTorch Version:", torch.__version__)
print("Device:", "CUDA" if torch.cuda.is_available() else "CPU")

PART A: LOADING AND CONFIGURING A PRE-TRAINED LLM

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
).to("cuda" if torch.cuda.is_available() else "cpu")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

model.eval()

print("Model Loaded Successfully!")
print("Model Name:", MODEL_NAME)
print("Device:", DEVICE)
print("Parameters:", sum(p.numel() for p in model.parameters()) / 1e6, "Million")

In [ ]:
sample_prompts = [
    "Explain what a Transformer model is.",
    "What is Artificial Intelligence?",
    "What is the capital of France?"
]

for prompt in sample_prompts:

    tokens = tokenizer(prompt, return_tensors="pt")

    print("\nPrompt:", prompt)
    print("Token IDs:", tokens["input_ids"])
    print("Number of Tokens:", tokens["input_ids"].shape[1])

In [ ]:
def generate_response(messages, temperature=0.7,
                      max_new_tokens=100, top_p=0.9):

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)

    gen_kwargs = {
        "max_new_tokens": max_new_tokens,
        "pad_token_id": tokenizer.eos_token_id
    }

    if temperature > 0:
        gen_kwargs.update(
            do_sample=True,
            temperature=temperature,
            top_p=top_p
        )
    else:
        gen_kwargs["do_sample"] = False

    start = time.time()

    with torch.no_grad():
        output = model.generate(**inputs, **gen_kwargs)

    elapsed = time.time() - start

    new_tokens = output[0][inputs["input_ids"].shape[1]:]

    reply = tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    ).strip()

    return reply, elapsed

In [ ]:
prompt = "Explain Artificial Intelligence in simple words."

settings = [
    {"temperature": 0.0, "max_new_tokens": 60},
    {"temperature": 0.7, "max_new_tokens": 100},
    {"temperature": 1.2, "max_new_tokens": 100}
]

for cfg in settings:

    messages = [
        {"role": "user", "content": prompt}
    ]

    reply, secs = generate_response(messages, **cfg)

    print("\nTemperature:", cfg["temperature"])
    print("Maximum Tokens:", cfg["max_new_tokens"])
    print("Response:", reply)
    print("Time:", round(secs, 2), "seconds")

PART B: IMPLEMENTING CHATBOT FUNCTIONALITY

In [ ]:
SYSTEM_PROMPT = "You are a helpful, concise AI assistant."

class Chatbot:

    def __init__(self, max_turns=6):

        self.history = []
        self.max_turns = max_turns

    def chat(self, user_input):

        self.history.append({
            "role": "user",
            "content": user_input
        })

        recent = self.history[-2 * self.max_turns:]

        messages = [
            {"role": "system", "content": SYSTEM_PROMPT}
        ] + recent

        reply, secs = generate_response(messages)

        self.history.append({
            "role": "assistant",
            "content": reply
        })

        return reply, secs

    def reset(self):

        self.history = []


bot = Chatbot()

print("Chatbot created successfully!")

In [ ]:
test_queries = [
    "Hi! My name is Asha.",
    "What is Machine Learning?",
    "What is my name?",
    "Explain Deep Learning."
]

for query in test_queries:

    reply, secs = bot.chat(query)

    print("\nUser:", query)
    print("Bot:", reply)
    print("Response Time:", round(secs, 2), "seconds")

In [ ]:
bot.reset()

reply, secs = bot.chat("My name is Krithika.")
print("Bot:", reply)

reply, secs = bot.chat("What is my name?")
print("Bot:", reply)

In [ ]:
bot.reset()

query = "Who won the 2050 FIFA World Cup?"

reply, secs = bot.chat(query)

print("User:", query)
print("Bot:", reply)
print("Response Time:", round(secs, 2), "seconds")

PART C: DEVELOPING AN INTERACTIVE INTERFACE

In [ ]:
MAX_TURNS = 6

def respond(message, history, temperature, max_new_tokens):

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT}
    ]

    messages += [
        {"role": m["role"], "content": m["content"]}
        for m in history[-2 * MAX_TURNS:]
    ]

    messages.append({
        "role": "user",
        "content": message
    })

    reply, secs = generate_response(
        messages,
        temperature,
        int(max_new_tokens)
    )

    return f"{reply}\n\n(Response time: {secs:.2f} seconds)"

In [ ]:
demo = gr.ChatInterface(
    fn=respond,
    title="Transformer LLM Chatbot",

    additional_inputs=[
        gr.Slider(
            0.0, 1.5,
            value=0.7,
            step=0.1,
            label="Temperature"
        ),

        gr.Slider(
            32, 512,
            value=256,
            step=32,
            label="Maximum New Tokens"
        )
    ],

    examples=[
        ["Explain Transformers in simple words.", 0.7, 256],
        ["What is Artificial Intelligence?", 0.5, 256],
        ["Write a Python program to reverse a string.", 0.3, 256]
    ]
)

print("Gradio interface created successfully!")

In [ ]:
demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://3d846bda577479ad41.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
